# 08 — Production Reef, Security, and Observability

**Mode:** Services  
**Level:** Advanced  
**Estimated time:** 40 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=wPXbCCjNBHw)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A service-backed production path: a native Spore makes a RabbitMQ round trip, a targeted SecureSpore is encrypted and verified, and a completed trace is stored in SQLite and exported to a real local OTLP collector. You will also observe a contained exporter failure and explicit shutdown.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(8, 'Production Reef, Security, and Observability')


## Prerequisites and setup

**Course prerequisites:** Complete `course-02-research-pipeline`, `course-04-parallel-agents`.

**Execution requirements:** Services. Start RabbitMQ and an OTLP HTTP collector, then set `RABBITMQ_URL` and `OTLP_HTTP_ENDPOINT`. Install the `secure` and `observability` extras.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Move a real Spore through the RabbitMQ backend.
- Encrypt, sign, serialize, decrypt, and verify a targeted SecureSpore.
- Finalize and inspect a trace before OTLP export.
- Contain exporter failure and close service resources.


## Mental model

A Reef transport changes **where** messages travel, not their Spore contract. Security protects the payload across that boundary. Tracing records the operation around it, and exporters copy completed spans to observability systems. Each layer has its own failure and shutdown behavior.


In [ ]:
show_flow(
('Producer', 'native Spore', 'agent'),
('RabbitMQ', 'AMQP transport', 'reef'),
('Consumer', 'verified payload', 'agent'),
('OTLP', 'completed span', 'provider'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Validate service configuration

The endpoints are required rather than silently skipped. The RabbitMQ URL is not displayed because it can contain credentials.


In [ ]:
import os
import tempfile
from datetime import datetime

values = require_env("RABBITMQ_URL", "OTLP_HTTP_ENDPOINT")
trace_directory = tempfile.TemporaryDirectory(prefix="praval-traces-")
trace_path = str(Path(trace_directory.name) / "notebook.db")
show_json(
    {"rabbitmq": "configured", "otlp_endpoint": values["OTLP_HTTP_ENDPOINT"]},
    "Service configuration",
)


### What just happened?

Both real services are configured, and trace storage has an isolated local path.

### Why this matters

Service certification should fail clearly when infrastructure is missing.


### Round-trip a native Spore through RabbitMQ

The backend serializes the Spore, publishes it through AMQP, consumes it, and reconstructs the message for an async handler.


In [ ]:
import asyncio

from praval.core.reef import Spore, SporeType
from praval.core.reef_backend import RabbitMQBackend


async def exercise_rabbitmq():
    backend = RabbitMQBackend()
    received = []
    delivered = asyncio.Event()

    async def handler(spore):
        received.append(spore)
        stage("consumer", "Spore received", spore.knowledge["type"], spore)
        delivered.set()

    try:
        await backend.initialize({
            "url": values["RABBITMQ_URL"], "exchange_name": "praval.notebook",
        })
        await backend.subscribe("agent.notebook-receiver", handler)
        spore = Spore(
            id="notebook-production-spore", spore_type=SporeType.KNOWLEDGE,
            from_agent="notebook-producer", to_agent="notebook-receiver",
            knowledge={"type": "production_probe", "ready": True},
            created_at=datetime.now(), schema_version="2.0",
            correlation_id="notebook-production",
        )
        stage("producer", "publish", "agent.notebook-receiver", spore)
        await backend.send(spore, "agent.notebook-receiver")
        await asyncio.wait_for(delivered.wait(), timeout=15)
        assert received[0].knowledge == spore.knowledge
        return received[0], backend.get_stats()
    finally:
        await backend.shutdown()
        stage("RabbitMQ", "shutdown", "connection closed")


### What just happened?

The async function owns backend initialization and a `finally` shutdown path; it has not executed yet.

### Why this matters

Keeping transport lifecycle in one boundary prevents leaked connections on failure.


### Execute and inspect transport delivery

Top-level `await` is supported in Jupyter. The returned Spore is the actual message received after the broker round trip.


In [ ]:
transported_spore, transport_stats = await exercise_rabbitmq()
assert transport_stats["spores_sent"] >= 1
show_spore(transported_spore, "Spore after RabbitMQ round trip")
show_json(transport_stats, "RabbitMQ backend statistics")


### What just happened?

A real broker accepted, routed, and returned the serialized message before the backend connection closed.

### Why this matters

Transport tests should assert payload integrity and cleanup, not merely connectivity.


### Protect a targeted Spore

Sender and recipient key managers create an encrypted, signed SecureSpore. The recipient verifies the signature before decrypting the knowledge.


In [ ]:
from praval.core.secure_spore import (
    SecureSpore, SecureSporeFactory, SporeKeyManager,
)

sender_keys = SporeKeyManager("secure-sender")
recipient_keys = SporeKeyManager("secure-recipient")
secure_factory = SecureSporeFactory(sender_keys)
protected = {"type": "private_probe", "value": "not visible on the wire"}
secure_spore = secure_factory.create_secure_spore(
    to_agent="secure-recipient", knowledge=protected,
    recipient_public_keys=recipient_keys.get_public_keys(),
)
wire_copy = SecureSpore.from_bytes(secure_spore.to_bytes())
decrypted = recipient_keys.decrypt_and_verify(
    wire_copy.encrypted_knowledge, wire_copy.nonce,
    wire_copy.knowledge_signature, wire_copy.sender_public_key,
    bytes(sender_keys.verify_key),
)
assert decrypted == protected
assert protected["value"].encode() not in secure_spore.to_bytes()


### What just happened?

The wire payload contains ciphertext and a signature. Only the intended recipient's private key could recover the original dictionary.

### Why this matters

Transport encryption and authenticity are separate from broker access control; production systems commonly need both.


In [ ]:
show_json(
    {
        "spore_id": secure_spore.id,
        "ciphertext_bytes": len(secure_spore.encrypted_knowledge),
        "signature_bytes": len(secure_spore.knowledge_signature),
        "decrypted_type": decrypted["type"],
    },
    "SecureSpore evidence",
    role="spore",
)


### Create and store a finalized trace

Observability is enabled for an isolated SQLite store. The context manager finalizes timing before storing the span.


In [ ]:
os.environ["PRAVAL_OBSERVABILITY"] = "on"
os.environ["PRAVAL_TRACES_PATH"] = trace_path

from praval.observability import SpanKind, get_trace_store, get_tracer
from praval.observability.config import reset_config
from praval.observability.storage.sqlite_store import reset_trace_store
from praval.observability.tracing.tracer import reset_tracer

reset_config()
reset_trace_store()
reset_tracer()
tracer = get_tracer()
with tracer.start_as_current_span(
    "praval.notebook.production", kind=SpanKind.CLIENT,
    attributes={"notebook": True},
) as span:
    span.add_event("rabbitmq.round_trip.complete")
    span.set_status("ok")

stored_spans = get_trace_store().get_trace(span.trace_id)
assert len(stored_spans) == 1
assert stored_spans[0]["end_time"] is not None
assert stored_spans[0]["duration_ms"] > 0


### What just happened?

The stored record already has an end time, duration, status, attributes, and event. It was written exactly once after the span ended.

### Why this matters

Exporting unfinished spans produces misleading durations and incomplete evidence.


In [ ]:
show_json(stored_spans[0], "Finalized SQLite span", role="provider")


### Export to OTLP and contain failure

The real collector must accept the span. A second exporter points at a closed loopback port to demonstrate that exporter failure returns `False` without changing application behavior.


In [ ]:
from praval.observability import OTLPExporter

exporter = OTLPExporter(values["OTLP_HTTP_ENDPOINT"])
exported = exporter.export_spans(stored_spans)
assert exported
stage("OTLP", "span exported", "praval.notebook.production")

failing_exporter = OTLPExporter("http://127.0.0.1:1/v1/traces")
failed_cleanly = failing_exporter.export_spans(stored_spans)
assert failed_cleanly is False
stage("OTLP", "failure contained", "application continued")
show_timeline()


### What just happened?

The primary export reached a real collector. The unreachable endpoint produced a contained exporter result rather than an application exception.

### Why this matters

Observability must report its own failure without becoming the business workflow's failure.


## Your turn

Add a child span inside the production span and verify both records share a trace ID while the child points to the parent's span ID.


In [ ]:
# with tracer.start_as_current_span("parent") as parent:
#     with tracer.start_as_current_span("child") as child:
#         assert child.trace_id == parent.trace_id
#         assert child.parent_span_id == parent.span_id


## Common mistake

**Mistake:** Closing the notebook without shutting down transport clients or removing local trace files.

**Correction:** Own external resources in `try/finally` or context managers and make cleanup an asserted stage of the workflow.


<details>
<summary>Under the hood</summary>

RabbitMQBackend maps channel names to AMQP routing and reconstructs native Spore fields. SecureSpore uses Curve25519 authenticated encryption and Ed25519 signatures. Tracer context finalizes a Span, restores context, then stores the completed object for console or OTLP export.

</details>


## Recap

- Transport changes location while preserving the Spore contract.
- SecureSpores protect targeted payload confidentiality and authenticity.
- Spans must finalize before storage and export.
- Transport and exporter failures need explicit containment and shutdown.


### Next lesson

Continue to `09_model_runtime.ipynb` to examine provider-neutral requests, capabilities, sync/async execution, streaming, usage, and structured output.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
reset_trace_store()
reset_tracer()
reset_config()
trace_directory.cleanup()
stage("observability", "shutdown", "temporary trace storage removed")
